# 5. Everything is a plugin: 교체와 해제

근거: [제품 페이지](https://www.deepseek.com/en/harness/), 저장소 docs/cordis-primer.md 및 docs/cordis-tutorial/02-lifecycle-and-effects.md.

**실행:** Python (Harness Lab) 커널에서 위에서 아래로 실행하세요. 각 노트북은 새 커널에서 독립 실행됩니다. 기본 셀은 외부 API를 호출하지 않습니다.

## “함수 목록”보다 중요한 것
Cordis는 서비스·이벤트·효과를 context에 등록하고, 플러그인이 내려갈 때 해당 등록을 해제합니다. 실제 Cordis는 의존성 상실과 비동기 정리까지 관리합니다. 아래 Python 구현은 동기 등록, 중복 거부, 역순 정리만 가르치는 작은 모델입니다. LangGraph 노드는 실행 단위이며 Cordis의 동적 plugin lifecycle과 일대일 대응하지 않습니다.

서비스 정의(Protocol), 제공자, 소비자를 분리하면 소비자 로직을 바꾸지 않고 구현을 교체할 수 있습니다. 원본은 agent loop 자체도 플러그인으로 교체 가능합니다.

In [1]:
from typing import Protocol
from contextlib import contextmanager, ExitStack
from langchain_core.runnables import RunnableLambda

class Search(Protocol):
    def find(self,query: str) -> str: ...
class LocalSearch:
    def find(self,query): return 'local:'+query
class CachedSearch:
    def find(self,query): return 'cache:'+query
class Context:
    def __init__(self): self.services={}
    def register(self,name,service):
        if name in self.services: raise ValueError('이미 등록됨: '+name)
        self.services[name]=service
        def dispose():
            if self.services.get(name) is service: del self.services[name]
        return dispose
    def require(self,name):
        if name not in self.services: raise LookupError('의존 서비스 없음: '+name)
        return self.services[name]

@contextmanager
def profile(ctx,provider):
    with ExitStack() as effects:
        effects.callback(ctx.register('search',provider))
        yield ctx
ctx=Context()
consumer=RunnableLambda(lambda q:ctx.require('search').find(q))
with profile(ctx,LocalSearch()):
    assert consumer.invoke('harness')=='local:harness'
    try: ctx.register('search',CachedSearch())
    except ValueError: print('중복 등록 거부')
    else: raise AssertionError('중복을 허용하면 안 됨')
assert ctx.services=={}
with profile(ctx,CachedSearch()): assert consumer.invoke('harness')=='cache:harness'
assert ctx.services=={}
try:
    with profile(ctx,LocalSearch()): raise RuntimeError('플러그인 초기화 실패')
except RuntimeError: pass
assert ctx.services=={}
print('교체·정상 종료·예외 종료 후 정리 확인')

중복 등록 거부
교체·정상 종료·예외 종료 후 정리 확인


## 정책은 실행 구현과 분리
around middleware는 next()로 다음 처리를 위임합니다. 관찰용 플러그인이 위임을 빼먹으면 도구가 사라진 것처럼 보입니다. 거부 정책의 short-circuit은 의도적입니다. 실제 원본은 waterfall과 최종 guard를 구분하므로 이 작은 chain을 보안 sandbox로 취급하면 안 됩니다.

In [2]:
audit=[]
def execute(request):
    audit.append('executed'); return {'result':request['text'].upper()}
def guard(request,next):
    if request['operation']!='read': return {'denied':True}
    return next(request)
def observe(request,next):
    audit.append('before')
    result=next(request)
    audit.append('after')
    return result
def pipeline(request): return observe(request,lambda req:guard(req,execute))
assert pipeline({'operation':'write','text':'x'})=={'denied':True}
assert 'executed' not in audit
audit.clear()
assert pipeline({'operation':'read','text':'hello'})['result']=='HELLO'
assert audit==['before','executed','after']
print(audit)

['before', 'executed', 'after']


## 직접 바꿔 보기

profile 종료 후 consumer를 호출하면 어떤 오류가 나야 할까요? dispose를 두 번 호출해도 안전하게 만들어야 하는 이유는 무엇인가요?

## 해설과 참고 답안

서비스가 없으므로 LookupError가 나야 합니다. 조용한 fallback은 잘못된 구성을 숨깁니다. 중복 정리나 부분 초기화 실패에 대비해 disposer를 멱등하게 만듭니다. 위 구현은 현재 등록 객체의 identity까지 비교해 다른 제공자의 등록을 지우지 않습니다.